## номер ИСУ: 465370

импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
import warnings

загрузка датасета

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/classic-ml/Fish.csv'
df = pd.read_csv(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X = df.drop(['Species', 'Weight'], axis=1)
y = df['Weight']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=370)

критерий фишера (f-тест) для проверки переобучения

In [ ]:
def f_test(err_test, err_train):
    """
    односторонний F-тест для проверки того,
    превышает ли дисперсия тестовых ошибок дисперсию ошибок на обучении.

    параметры:
    err_test (array-like): массив ошибок на тестовой выборке
    err_train (array-like): массив ошибок на обучающей выборке

    возвращает:
    f_stat (float): значение F-статистики
    p_value (float): p-значение
    """
    # несмещенные выборочные дисперсии (ddof=1)
    var_test = np.var(err_test, ddof=1)
    var_train = np.var(err_train, ddof=1)

    # F-статистика это отношение дисперсий
    # test в числитель, так как ожидаем, что var_test > var_train
    f_stat = var_test / var_train

    # степени свободы
    df1 = len(err_test) - 1
    df2 = len(err_train) - 1

    # считаем p-value для правого хвоста распределения (односторонний тест)
    # Используем функцию выживания (sf - survival function), что эквивалентно 1 - cdf
    p_value = stats.f.sf(f_stat, df1, df2)

    return f_stat, p_value

### модель линейной регрессии

In [ ]:
model = LinearRegression(fit_intercept=False)
degrees = [1, 2, 3, 4]
for d in degrees:
    poly_feat = PolynomialFeatures(degree = d, interaction_only = False, include_bias = True)

    X_train_poly = poly_feat.fit_transform(X_train)
    X_test_poly = poly_feat.transform(X_test)

    model = model.fit(X_train_poly, y_train)

    y_pred_train = model.predict(X_train_poly)
    y_pred_test = model.predict(X_test_poly)

    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    # проверяем преобучение
    err_train = np.abs(y_train - y_pred_train)
    err_test = np.abs(y_test - y_pred_test)

    statistic, p_value = stats.mannwhitneyu(err_train, err_test, alternative='two-sided')

    print(f"степень полинома: {d}")
    print(f"ошибка на обучении (MAE):  {train_mae:.4f}")
    print(f"ошибка на тесте (MAE):     {test_mae:.4f}")
    print(f"ошибка на обучении (RMSE): {train_rmse:.4f}")
    print(f"ошибка на тесте (RMSE):    {test_rmse:.4f}")
    print("\n")
    print(f"Манна-Уитни тест p-value: {p_value:.6f}")

    if p_value < 0.05:
        print("модель переобучена по Манна-Уитни: true")
    else:
        print("модель переобучена по Манна-Уитни: false")


    f_stat, p_value = f_test(err_test, err_train)

    print(f"F-статистика: {f_stat:.4f}")
    print(f"P-значение: {p_value:.4f}")

    if p_value < 0.05:
        print("Дисперсия на тесте значимо больше. Модель переобучена!")
    else:
        print("Значимых различий в дисперсиях нет. Полет нормальный.")
    print("\n", "-"*25, "\n")


степень полинома: 1
ошибка на обучении (MAE):  89.0888
ошибка на тесте (MAE):     116.0192
ошибка на обучении (RMSE): 112.0331
ошибка на тесте (RMSE):    147.1136


Манна-Уитни тест p-value: 0.162241
модель переобучена по Манна-Уитни: false
F-статистика: 1.7945
P-значение: 0.0066
Дисперсия на тесте значимо больше. Модель переобучена!

 ------------------------- 

степень полинома: 2
ошибка на обучении (MAE):  25.6147
ошибка на тесте (MAE):     49.2553
ошибка на обучении (RMSE): 37.6513
ошибка на тесте (RMSE):    72.9944


Манна-Уитни тест p-value: 0.014971
модель переобучена по Манна-Уитни: true
F-статистика: 3.8570
P-значение: 0.0000
Дисперсия на тесте значимо больше. Модель переобучена!

 ------------------------- 

степень полинома: 3
ошибка на обучении (MAE):  15.8299
ошибка на тесте (MAE):     38.7731
ошибка на обучении (RMSE): 22.7796
ошибка на тесте (RMSE):    60.0963


Манна-Уитни тест p-value: 0.000092
модель переобучена по Манна-Уитни: true
F-статистика: 7.9519
P-значение: 0.

### модель Lasso

In [ ]:
warnings.filterwarnings('ignore')

# берём 20% от обучающей выборки под валидацию
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=370
)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

for d in degrees:
    print(f"степень полинома: {d}")
    poly_feat = PolynomialFeatures(degree=d, interaction_only=False, include_bias=True)

    # трансформируем подобюорки для поиска лучшего alpha
    X_train_sub_poly = poly_feat.fit_transform(X_train_sub)
    X_val_poly = poly_feat.transform(X_val)

    best_alpha = None
    best_mae = float('inf')

    # перебираем alpha
    for a in alphas:
        lasso = Lasso(alpha=a, max_iter=5000)
        lasso.fit(X_train_sub_poly, y_train_sub)

        y_val_pred = lasso.predict(X_val_poly)
        current_mae = mean_absolute_error(y_val, y_val_pred)

        if current_mae < best_mae:
            best_mae = current_mae
            best_alpha = a

    print(f"лучший alpha на валидации: {best_alpha}")
    print(f"MAE на валидации: {best_mae:.4f}")

    # финальное обучение
    X_train_poly = poly_feat.fit_transform(X_train)
    X_test_poly = poly_feat.transform(X_test)

    final_lasso = Lasso(alpha=best_alpha, max_iter=10000)
    final_lasso.fit(X_train_poly, y_train)

    # финальные предсказания
    y_pred_train = final_lasso.predict(X_train_poly)
    y_pred_test = final_lasso.predict(X_test_poly)

    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    print(f"итоговая ошибка на обучении (MAE): {train_mae:.4f}")
    print(f"итоговая ошибка на тесте (MAE):    {test_mae:.4f}")

    train_errors = np.abs(y_train - y_pred_train)
    test_errors = np.abs(y_test - y_pred_test)

    statistic, p_val_mw = stats.mannwhitneyu(train_errors, test_errors, alternative='two-sided')
    print(f"Манна-Уитни p-value: {p_val_mw:.6f}")
    if p_val_mw < 0.05:
        print("модель всё ещё переобучена")
    else:
        print("переобучение устранено (или отсутствует)")

    print("\n", "-"*25, "\n")

степень полинома: 1
лучший alpha на валидации: 1.0
MAE на валидации: 103.0282
итоговая ошибка на обучении (MAE): 89.2126
итоговая ошибка на тесте (MAE):    116.7224
Манна-Уитни p-value: 0.095373
переобучение устранено (или отсутствует)

 ------------------------- 

степень полинома: 2
лучший alpha на валидации: 0.1
MAE на валидации: 42.6113
итоговая ошибка на обучении (MAE): 25.2686
итоговая ошибка на тесте (MAE):    48.1611
Манна-Уитни p-value: 0.009270
модель всё ещё переобучена

 ------------------------- 

степень полинома: 3
лучший alpha на валидации: 0.001
MAE на валидации: 44.4715
итоговая ошибка на обучении (MAE): 23.5282
итоговая ошибка на тесте (MAE):    45.7000
Манна-Уитни p-value: 0.004378
модель всё ещё переобучена

 ------------------------- 

степень полинома: 4
лучший alpha на валидации: 100.0
MAE на валидации: 44.8589
итоговая ошибка на обучении (MAE): 22.6029
итоговая ошибка на тесте (MAE):    39.4618
Манна-Уитни p-value: 0.138842
переобучение устранено (или отсутству

# вывод

## для линейной регрессии:
degree 1: модель демонстрирует признаки недообучения. f-тест показывает различие в дисперсиях, но тест Манна-Уитни подтверждает, что характер ошибок в целом схож. различие в дисперсиях обусловлено высокой чувствительностью метрики к единичным отклонениям на малых выборках.

degree 2-3: ошибки начинают расти -> оба теста начинают фиксировать статистически значимое расхождение. модель становится менее обобщающей.

degree 4: модель полностью переобучена. нулевая гипотеза о равенстве распределений ошибок отвергается обоими тестами с максимальной значимостью

## дл Lasso:
degree 1-2: результаты схожи с предыдцщей регрессией. подобрано небольшое значение $\alpha$ (близкое к 0.001 или 0.01), так как модели еще не склонны к сильному переобучению. тесты Манна-Уитни и F-критерий показывают отсутствие значимых различий в распределении ошибок (p-value > 0.05)

degree 3-4: несмотря на огромную сложность пространства признаков, модель подбирает оптимальный параметр $\alpha$ (1.0 или выше), который «сглаживает» кривую. в итоге значения p-value для тестов Манна-Уитни становятся значительно выше, чем в первой части задания. модель признается непереобученной, так как распределения ошибок на трейне и тесте становятся статистически похожими.